# Unpooling Strategies — Section 5.1

In autoencoders and segmentation networks (e.g. SegNet), the encoder uses **max pooling** to downsample feature maps. The decoder must **upsample** back to the original resolution. Three common strategies:

| Strategy | Description | Needs stored indices? |
|---|---|---|
| **Max-unpooling** | Places each pooled value back at the exact position of the original max | Yes |
| **Bed-of-nails** | Places pooled value at top-left (0,0) of each block; zeros elsewhere | No |
| **Nearest-neighbor** | Copies pooled value to all 4 cells of each block | No |

### Example (4×4 → 2×2 → 4×4)

Given a 4×4 feature map, max pooling with 2×2 non-overlapping windows produces a 2×2 result.  
The decoder then unpools back to 4×4 using one of the three strategies.

**Key insight:** Max-unpooling produces sparse outputs but preserves precise spatial information. Nearest-neighbor is dense but blurry. Bed-of-nails is a compromise — structured but still sparse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def max_pool2x2(x):
    """Returns pooled values and indices (row,col within 2x2 block)."""
    h, w = x.shape
    pooled = np.zeros((h//2, w//2))
    indices = np.zeros((h//2, w//2, 2), dtype=int)
    for i in range(h//2):
        for j in range(w//2):
            block = x[2*i:2*i+2, 2*j:2*j+2]
            idx = np.unravel_index(np.argmax(block), (2,2))
            pooled[i,j] = block[idx]
            indices[i,j] = idx
    return pooled, indices

def unpool_max(pooled, indices):
    h, w = pooled.shape
    out = np.zeros((h*2, w*2))
    for i in range(h):
        for j in range(w):
            ri, ci = indices[i,j]
            out[2*i+ri, 2*j+ci] = pooled[i,j]
    return out

def unpool_bed_of_nails(pooled):
    h, w = pooled.shape
    out = np.zeros((h*2, w*2))
    for i in range(h):
        for j in range(w):
            out[2*i, 2*j] = pooled[i,j]
    return out

def unpool_nearest(pooled):
    return np.repeat(np.repeat(pooled, 2, axis=0), 2, axis=1)

def draw_unpooling(seed=42):
    rng = np.random.default_rng(seed)
    x = rng.integers(1, 10, (4, 4)).astype(float)
    pooled, indices = max_pool2x2(x)
    
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
    
    def show_grid(ax, data, title, vmin=0, vmax=9, cmap='Blues'):
        ax.imshow(data, vmin=vmin, vmax=vmax, cmap=cmap, aspect='equal')
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                ax.text(j, i, f'{data[i,j]:.0f}', ha='center', va='center',
                        fontsize=13, fontweight='bold',
                        color='white' if data[i,j] > 5 else 'black')
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
    
    show_grid(axes[0], x, 'Input (4x4)')
    show_grid(axes[1], pooled, 'Max-pooled (2x2)\n(indices stored)')
    show_grid(axes[2], unpool_max(pooled, indices), 'Max-unpooling\n(use stored indices)')
    show_grid(axes[3], unpool_bed_of_nails(pooled), 'Bed-of-nails\n(top-left only)')
    show_grid(axes[4], unpool_nearest(pooled), 'Nearest-neighbor\n(fill 2x2 block)')
    
    plt.suptitle('3 Unpooling Strategies on a 4x4 Feature Map', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

widgets.interact(draw_unpooling,
    seed=widgets.IntSlider(min=0, max=20, step=1, value=42, description='Random seed', continuous_update=False),
)